In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import kmeans_plusplus
from scipy.io import savemat, loadmat
import scipy.io as sio


In [2]:
# ------------------------------------------------
# 1. Dataset configuration
# ------------------------------------------------
# NOTE: Update feature_cols per dataset to match your CSV schema.

DATASET_CONFIG = {
    "adult": {
        "csv_path": "data/adult_preprocessed.csv",
        "feature_cols": [
            "age", "final-weight", "education-num", "marital-status",
            "occupation", "relationship", "capital-gain",
            "hours-per-week", "native-country", "income"
        ],
        "sensitive_col": "sex",
        "k": 7,
        "tag": "adult",   # used in file names
    },
    
    "student": {
        "csv_path": "data/student_preprocessed.csv",         # <-- FILL THIS
        "feature_cols": [
            'school', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
           'Mjob', 'Fjob', 'traveltime', 'studytime', 'failures', 'schoolsup',
           'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet',
           'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health',
           'absences'
        ],
        "sensitive_col": "sex",                    # <-- Example
        "k": 5,
        "tag": "student"
    },
    "bank": {
        "csv_path": "data/bank_preprocessed.csv",
        "feature_cols": [
            'age', 'balance', 'duration'
        ],
        "sensitive_col": "marital",         # <-- Example
        "k": 5,
        "tag": "bank"
    },
    "credit": {
        "csv_path": "data/credit_preprocessed.csv",     # <-- FILL THIS
        "feature_cols": [
            'LIMIT_BAL', 'SEX', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
            'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2',
            'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
        ],
        "sensitive_col": "MARRIAGE",                    # <-- Example, change if needed
        "k": 5,
        "tag": "credit"
    }
}


In [3]:
# ------------------------------------------------
# 2. Utility functions
# ------------------------------------------------

def load_dataset(dataset_name: str):
    """Load and standardize the dataset, and build sensitive group masks."""
    cfg = DATASET_CONFIG[dataset_name]
    df = pd.read_csv(cfg["csv_path"])
    X = df[cfg["feature_cols"]].values
    s = df[cfg["sensitive_col"]].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    unique_vals = np.unique(s[~pd.isna(s)])

    print(f"Sensitive column '{cfg['sensitive_col']}' unique values:", unique_vals)

    if len(unique_vals) != 2:
        raise ValueError(
            f"Sensitive attribute must be binary. Found {len(unique_vals)} values."
        )

    val_A, val_B = unique_vals

    A_mask = (s == val_A)
    B_mask = (s == val_B)

    XA = X_scaled[A_mask]
    XB = X_scaled[B_mask]

    print(f"Dataset '{dataset_name}': {len(XA)} in group A ({val_A}), "
        f"{len(XB)} in group B ({val_B})")
    # We assume sensitive attribute is binary and >0 means group A
    #A_mask = (s > 0)
    #B_mask = ~A_mask

    print(f"[{dataset_name}] n = {len(X_scaled)}")
    print(f"[{dataset_name}] |A| = {A_mask.sum()}, |B| = {B_mask.sum()}")
    assert A_mask.sum() + B_mask.sum() == len(X_scaled)

    return X_scaled, A_mask, B_mask, cfg


def assign_points(X, C):
    """Assign each point in X to its closest center in C."""
    d2 = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    return d2.argmin(axis=1)


def kmeans_loss(X, C, a):
    """Standard k-means loss (mean squared distance to assigned centers)."""
    return ((X - C[a]) ** 2).sum() / len(X)


def delta_group(X, C, a, group_mask):
    """Sum of squared distances for points in one demographic group."""
    idx = np.where(group_mask)[0]
    if len(idx) == 0:
        return 0.0
    return ((X[idx] - C[a[idx]]) ** 2).sum()


def phi_max(X, C, a, A_mask, B_mask):
    """
    Compute average distance for each group and return:
    (worst_group_cost, 'A' or 'B').
    """
    nA = A_mask.sum()
    nB = B_mask.sum()
    if nA == 0 or nB == 0:
        raise ValueError("Both groups must be present.")

    dA = delta_group(X, C, a, A_mask) / nA
    dB = delta_group(X, C, a, B_mask) / nB
    if dA >= dB:
        return dA, "A"
    else:
        return dB, "B"


def grad_centers(X, C, a, A_mask, B_mask, lam):
    """
    Gradient of the Social Fair k-means objective with respect to centers.
    Includes:
      - standard k-means gradient
      - plus λ * fairness gradient for the worst-off group.
    """
    n = len(X)
    k, d = C.shape

    Gk = np.zeros_like(C)      # k-means gradient
    Gfair = np.zeros_like(C)   # fairness gradient

    # --- k-means gradient ---
    for j in range(k):
        idx = np.where(a == j)[0]
        if idx.size == 0:
            continue
        diff = C[j] - X[idx]          # (#idx, d)
        Gk[j] = (2.0 / n) * diff.sum(axis=0)

    # --- fairness gradient ---
    if lam != 0.0:
        _, branch = phi_max(X, C, a, A_mask, B_mask)

        if branch == "A":
            Ng = int(A_mask.sum())
            if Ng > 0:
                for j in range(k):
                    idx = np.where((a == j) & A_mask)[0]
                    if idx.size == 0:
                        continue
                    diff = C[j] - X[idx]
                    Gfair[j] += (2.0 / Ng) * diff.sum(axis=0)
        else:
            Ng = int(B_mask.sum())
            if Ng > 0:
                for j in range(k):
                    idx = np.where((a == j) & B_mask)[0]
                    if idx.size == 0:
                        continue
                    diff = C[j] - X[idx]
                    Gfair[j] += (2.0 / Ng) * diff.sum(axis=0)

    return Gk + lam * Gfair


def update_centroids_analytic(X, a, k, C_prev):
    """Standard Lloyd centroid update, keeping empty clusters unchanged."""
    C_new = C_prev.copy()
    for j in range(k):
        idx = np.where(a == j)[0]
        if idx.size > 0:
            C_new[j] = X[idx].mean(axis=0)
        else:
            # Keep the old center if the cluster is empty
            C_new[j] = C_prev[j]
    return C_new


def train_fair_kmeans(X, A_mask, B_mask, C0, lam=1.0,
                      iters=500, inner_gd=5, lr=0.05):
    """
    Social Fair k-means training.

    If lam = 0, this reduces to standard Lloyd's algorithm.
    Otherwise, we alternate between hard assignments and several
    gradient-descent steps on the centers.
    """
    C = C0.copy()
    history = []
    k = C.shape[0]
    use_lloyd = (lam <= 1e-12)

    for t in range(iters):
        # (a) hard assignments
        a = assign_points(X, C)

        if use_lloyd:
            C = update_centroids_analytic(X, a, k, C)
        else:
            for _ in range(inner_gd):
                G = grad_centers(X, C, a, A_mask, B_mask, lam)
                C = C - lr * G

        km = kmeans_loss(X, C, a)
        if not use_lloyd:
            phi_val, _ = phi_max(X, C, a, A_mask, B_mask)
        else:
            phi_val = 0.0
        history.append({"iter": t, "kmeans_loss": km, "phi": phi_val})

    return C, history

In [4]:
def compute_pairwise_fairness(X, XA, XB, centroids):
    """
    Separation-style fairness (counterfactual hyperplane distance),
    kept here only as an additional diagnostic.
    """
    def get_squared_distance_to_hyperplane(x, m1, m2):
        m = 0.5 * (m1 + m2)
        v = m2 - m1
        v = v / np.linalg.norm(v)
        return ((x - m) @ v) ** 2

    fairness_A = []
    fairness_B = []
    for x in XA:
        dists = np.linalg.norm(centroids - x, axis=1)
        m1_idx = np.argmin(dists)
        m1 = centroids[m1_idx]
        dists[m1_idx] = np.inf
        m2 = centroids[np.argmin(dists)]
        fairness_A.append(get_squared_distance_to_hyperplane(x, m1, m2))

    for x in XB:
        dists = np.linalg.norm(centroids - x, axis=1)
        m1_idx = np.argmin(dists)
        m1 = centroids[m1_idx]
        dists[m1_idx] = np.inf
        m2 = centroids[np.argmin(dists)]
        fairness_B.append(get_squared_distance_to_hyperplane(x, m1, m2))

    mu_A = np.mean(fairness_A)
    mu_B = np.mean(fairness_B)
    return mu_A, mu_B, min(mu_A, mu_B)

In [5]:
def run_unfair_lloyd(X, C0, iters=200, tol=1e-10):
    """Run standard (unfair) Lloyd k-means starting from centers C0."""
    C = C0.copy()
    for _ in range(iters):
        a = assign_points(X, C)
        C_new = update_centroids_analytic(X, a, C.shape[0], C)
        if np.linalg.norm(C_new - C) < tol:
            C = C_new
            break
        C = C_new
    return C

In [6]:
# ------------------------------------------------
# 3. Main Social Fair training loop
# ------------------------------------------------

def run_social_fair_kmeans_pipeline(dataset_name: str,
                                    lambda2_list=None,
                                    seeds=None,
                                    iters_fair=500,
                                    inner_gd=5,
                                    lr=0.5):
    """
    Train Social Fair k-means on the chosen dataset for a list of λ values
    and random seeds.

    Returns:
        metrics: dict with entries:
          - 'max_fair_dict'
          - 'fairness_gaps'
          - 'kmeans_costs'
          - 'unfair_kmeans_costs'
          - 'min_fair_dict'
        cfg: dataset configuration
    """
    if lambda2_list is None:
        lambda2_list = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    if seeds is None:
        seeds = list(range(10))

    X_input, A_mask, B_mask, cfg = load_dataset(dataset_name)
    k = cfg["k"]
    tag = cfg["tag"]

    # Metric containers
    max_fair_dict_lambda2 = {lam: [] for lam in lambda2_list}
    fairness_gaps_lambda2 = {lam: [] for lam in lambda2_list}
    kmeans_costs_lambda2 = {lam: [] for lam in lambda2_list}
    unfair_kmeans_costs_lambda2 = {lam: [] for lam in lambda2_list}
    min_fair_dict_lambda2 = {lam: [] for lam in lambda2_list}
    sep_fairness_gaps_lambda2 = {lam: [] for lam in lambda2_list}
    # Directories for saving
    root = Path(".")
    init_dir = root / "initial_centers_seeds"
    fair_dir = root / "clustering_file_with_seed" / "fair"
    unfair_dir = root / "clustering_file_with_seed" / "unfair"
    unfair_dir =  root / "clustering_file_with_seed_lambda"
    for d in [init_dir, fair_dir, unfair_dir]:
        d.mkdir(parents=True, exist_ok=True)
    metrics_root = Path("social_metrics")  
    metrics_root.mkdir(parents=True, exist_ok=True)

   
    for seed in seeds:
        print(f"\n=== Dataset: {dataset_name}, seed = {seed} ===")
        
        init_centers, _ = kmeans_plusplus(X_input, n_clusters=k,
                                          random_state=seed)
         
        savemat(
            (init_dir / f"{tag}_seed_{seed}_k_{k}_centers.mat").as_posix(),
            {
                "centers": init_centers.astype(np.float64),
                "seed": int(seed),
                "k": int(k),
                "d": int(X_input.shape[1]),
            },
            do_compression=True,
        )

       
        C_unfair = run_unfair_lloyd(X_input, init_centers.copy())
        a_unfair = assign_points(X_input, C_unfair)
        km_unfair = kmeans_loss(X_input, C_unfair, a_unfair)

        labels_unfair_1based = (a_unfair.astype(np.int32) + 1)
        savemat(
            (
                unfair_dir
                / f"clustering_{tag}_seed_{seed}_lambda_0.0.mat"
            ).as_posix(),
            {
                "female_labels": labels_unfair_1based[B_mask],
                "male_labels": labels_unfair_1based[A_mask],
                "labels_all": labels_unfair_1based,
                "lambda2": 0.0,
                "seed": int(seed),
                "k": int(k),
            },
            do_compression=True,
        )

        XA = X_input[A_mask]
        XB = X_input[B_mask]

        
        for lam2 in lambda2_list:
            
            unfair_kmeans_costs_lambda2[lam2].append(km_unfair)
             

            if lam2 == 0.0:
                C_final = C_unfair.copy()
                a_final = a_unfair.copy()
                km_fair = km_unfair
            else:
                C_final, _ = train_fair_kmeans(
                    X_input,
                    A_mask,
                    B_mask,
                    C0=C_unfair.copy(),
                    lam=lam2,
                    iters=iters_fair,
                    inner_gd=inner_gd,
                    lr=lr,
                )
                a_final = assign_points(X_input, C_final)
                km_fair = kmeans_loss(X_input, C_final, a_final)

            # Social fairness metrics
            valA = ((X_input[A_mask] - C_final[a_final[A_mask]]) ** 2).sum() / A_mask.sum()
            valB = ((X_input[B_mask] - C_final[a_final[B_mask]]) ** 2).sum() / B_mask.sum()
            max_fair = max(valA, valB)
            diff_abs = abs(valA - valB)

            # Separation-style fairness (optional)
            mu_A_sep, mu_B_sep, min_fair = compute_pairwise_fairness(
                X_input, XA, XB, C_final
            )
            sep_gap = abs(mu_A_sep - mu_B_sep)
            sep_fairness_gaps_lambda2[lam2].append(sep_gap)

            kmeans_costs_lambda2[lam2].append(km_fair)
            max_fair_dict_lambda2[lam2].append(max_fair)
            fairness_gaps_lambda2[lam2].append(diff_abs)
            min_fair_dict_lambda2[lam2].append(min_fair)

            labels_fair_1based = (a_final.astype(np.int32) + 1)
            savemat(
                (
                    fair_dir
                    / f"clustering_{tag}_seed_{seed}_(1-lambda)_{lam2:.1f}.mat"
                ).as_posix(),
                {
                    "female_labels": labels_fair_1based[B_mask],
                    "male_labels": labels_fair_1based[A_mask],
                    "labels_all": labels_fair_1based,
                    "lambda2": float(lam2),
                    "seed": int(seed),
                    "k": int(k),
                },
                do_compression=True,
            )

    for lam2 in lambda2_list:
        print(
            f"λ={lam2:.1f} | "
            f"km_unfair(mean)={np.mean(unfair_kmeans_costs_lambda2[lam2]):.6f} | "
            f"km_fair(mean)={np.mean(kmeans_costs_lambda2[lam2]):.6f} | "
            f"gap(mean)={np.mean(fairness_gaps_lambda2[lam2]):.6f} | "
            f"min_fair(mean)={np.mean(min_fair_dict_lambda2[lam2]):.6f}"
            f"sep_gap(mean)={np.mean(sep_fairness_gaps_lambda2[lam2]):.6f} | "
        )

    metrics = {
        "max_fair_dict": max_fair_dict_lambda2,
        "fairness_gaps": fairness_gaps_lambda2,
        "sep_fairness_gaps": sep_fairness_gaps_lambda2,
        "kmeans_costs": kmeans_costs_lambda2,
        "unfair_kmeans_costs": unfair_kmeans_costs_lambda2,
        "min_fair_dict": min_fair_dict_lambda2
    }
 

    np.savez(
        metrics_root / f"social_metrics_{dataset_name}.npz",
        max_fair_dict=max_fair_dict_lambda2,
        fairness_gaps=fairness_gaps_lambda2,
        sep_fairness_gaps=sep_fairness_gaps_lambda2,
        kmeans_costs=kmeans_costs_lambda2,
        unfair_kmeans_costs=unfair_kmeans_costs_lambda2,
        min_fair_dict=min_fair_dict_lambda2,
        lambda2_list=np.array(lambda2_list),
        seeds=np.array(seeds),
        k=int(k),
    )
    return metrics, cfg, lambda2_list


In [7]:
# ------------------------------------------------
# 4. Load Fair-Lloyd .mat results (optional, if you have them)
# ------------------------------------------------

def load_fair_lloyd_stats(dataset_tag, lambda2_list, seeds):
    """
    Load Fair-Lloyd results from MATLAB .mat files, if available.
    Expects files in: cost_seeds/full_results_seed_{seed}_lambda_{lam}_{tag}.mat

    Returns:
      stats dict OR None if no files were found.
    """
    stats_root = Path("fair_lloyd_metrics")  
    stats_root.mkdir(parents=True, exist_ok=True)
    cfg = DATASET_CONFIG[dataset_tag]
    k = cfg["k"]
    def f(x):
        return float(np.asarray(x).squeeze())

    n_lam = len(lambda2_list)
    n_seeds = len(seeds)

    costs_unfair_F = np.zeros((n_lam, n_seeds))
    costs_unfair_M = np.zeros((n_lam, n_seeds))
    costs_social_F = np.zeros((n_lam, n_seeds))
    costs_social_M = np.zeros((n_lam, n_seeds))

    unfair_diff = np.zeros((n_lam, n_seeds))
    fair_diff = np.zeros((n_lam, n_seeds))

    k_means_loss_unfair = np.zeros((n_lam, n_seeds))
    k_means_loss_fair = np.zeros((n_lam, n_seeds))

    maxfair_unfair = np.full((n_lam, n_seeds), np.nan)
    maxfair_fair = np.full((n_lam, n_seeds), np.nan)
    sep_min_fair = np.full((n_lam, n_seeds), np.nan)

    any_found = False

    for i, lam in enumerate(lambda2_list):
        lam_str = f"{lam:.1f}"
        for j, seed in enumerate(seeds):
            result_path = f"cost_seeds/full_results_seed_{seed}_k_{k}_{dataset_tag}.mat"
            if not os.path.exists(result_path):
                # you can uncomment if you want to see missing files:
                # print(f"[Fair-Lloyd] Missing: seed={seed}, λ={lam_str}")
                continue

            any_found = True
            import scipy.io as sio

            cost = sio.loadmat("cost_seeds/full_results_seed_9_k_5_credit.mat")
            
            cost = sio.loadmat(result_path)
            
            sep_min_fair[i, j] = f(cost["sepMinFair"])
            costs_unfair_F[i, j] = f(cost["costUnfair"][0][0])
            costs_unfair_M[i, j] = f(cost["costUnfair"][1][0])
            costs_social_F[i, j] = f(cost["costFair"][0][0])
            costs_social_M[i, j] = f(cost["costFair"][1][0])

            unfair_diff[i, j] = abs(costs_unfair_F[i, j] - costs_unfair_M[i, j])
            fair_diff[i, j] = abs(costs_social_F[i, j] - costs_social_M[i, j])

            k_means_loss_unfair[i, j] = f(cost["kmUnfair"])
            k_means_loss_fair[i, j] = f(cost["kmFair"])
           
            if all(k in cost for k in ("kmUnfairG1", "kmUnfairG2", "kmFairG1", "kmFairG2")):
                maxfair_unfair[i, j] = max(f(cost["kmUnfairG1"]), f(cost["kmUnfairG2"]))
                maxfair_fair[i, j] = max(f(cost["kmFairG1"]), f(cost["kmFairG2"]))

    if not any_found:
        print("[Fair-Lloyd] No .mat results found – skipping Fair-Lloyd curves.")
        return None

    stats = {
        "unfair_diff": unfair_diff,
        "fair_diff": fair_diff,
        "k_means_loss_unfair": k_means_loss_unfair,
        "k_means_loss_fair": k_means_loss_fair,
        "maxfair_unfair": maxfair_unfair,
        "maxfair_fair": maxfair_fair,
        "sep_min_fair": sep_min_fair,
    }
    np.savez(
    stats_root / f"fair_lloyd_stats_{dataset_tag}.npz",
    unfair_diff=unfair_diff,
    fair_diff=fair_diff,
    k_means_loss_unfair=k_means_loss_unfair,
    k_means_loss_fair=k_means_loss_fair,
    maxfair_unfair=maxfair_unfair,
    maxfair_fair=maxfair_fair,
    sep_min_fair=sep_min_fair,
    lambda2_list=np.array(lambda2_list),
    seeds=np.array(seeds),
    )
     
    return stats


In [ ]:
# ------------------------------------------------
# 5. Example usage
# ------------------------------------------------

# Choose one of: "adult", "credit", "bank", "student"
dataset_name = "student"

metrics_social, cfg_social, lambda_list = run_social_fair_kmeans_pipeline(
    dataset_name,
    lambda2_list=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    seeds=range(10),
    iters_fair=500,
    inner_gd=1,
    lr=0.01,
)

fair_lloyd_stats = load_fair_lloyd_stats(
    cfg_social["tag"], lambda_list, seeds=range(10)
)


Sensitive column 'sex' unique values: [0 1]
Dataset 'student': 359 in group A (0), 241 in group B (1)
[student] n = 600
[student] |A| = 359, |B| = 241

=== Dataset: student, seed = 0 ===

=== Dataset: student, seed = 1 ===

=== Dataset: student, seed = 2 ===

=== Dataset: student, seed = 3 ===

=== Dataset: student, seed = 4 ===

=== Dataset: student, seed = 5 ===

=== Dataset: student, seed = 6 ===

=== Dataset: student, seed = 7 ===

=== Dataset: student, seed = 8 ===

=== Dataset: student, seed = 9 ===
λ=0.0 | km_unfair(mean)=22.249377 | km_fair(mean)=22.249377 | gap(mean)=1.262959 | min_fair(mean)=1.445687sep_gap(mean)=0.439489 | 
λ=0.2 | km_unfair(mean)=22.249377 | km_fair(mean)=22.261312 | gap(mean)=0.952781 | min_fair(mean)=1.426021sep_gap(mean)=0.476577 | 
λ=0.4 | km_unfair(mean)=22.249377 | km_fair(mean)=22.286121 | gap(mean)=0.717426 | min_fair(mean)=1.402104sep_gap(mean)=0.511607 | 
λ=0.6 | km_unfair(mean)=22.249377 | km_fair(mean)=22.315418 | gap(mean)=0.536074 | min_fair(m